In [1]:
# Import libraries
import numpy as np
import pickle
import mne
import os
import ipywidgets as widgets
import matplotlib.pyplot as plt
import sklearn
%matplotlib qt
# set size of raw browser
mne.utils.config.set_config('MNE_BROWSE_RAW_SIZE','16.0,8.0')

In [2]:
paths = [os.path.join("open_science_data", p) for p in os.listdir("open_science_data") if p.endswith(".gdf")]
print(paths)


['open_science_data/V06.gdf', 'open_science_data/V04.gdf', 'open_science_data/V05.gdf', 'open_science_data/V01.gdf', 'open_science_data/V02.gdf', 'open_science_data/V03.gdf']


In [4]:
data_dir = {}
for p in paths:
    tag = p[-7:-4]
    try:
        data_dir[tag] = mne.io.read_raw_gdf(p, preload=True)

    except:
        print(tag)
    




Extracting GDF parameters from open_science_data/V06.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
FP1, FP2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T7, T8, P7, P8, Fz, Cz, Pz, A1, A2, AFz, CPz, POz
Creating raw.info structure...
Reading 0 ... 369983  =      0.000 ...  1479.932 secs...
Extracting GDF parameters from open_science_data/V04.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
FP1, FP2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T7, T8, P7, P8, Fz, Cz, Pz, M1, M2, AFz, CPz, POz
Creating raw.info structure...
Reading 0 ... 369951  =      0.000 ...  1479.804 secs...
Extracting GDF parameters from open_science_data/V05.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
FP1, FP2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T7, T8, P7, P8, Fz, Cz, Pz, A1, A2, A

In [29]:
from mne.preprocessing import annotate_muscle_zscore

raw = data_dir["V06"]

def preprocess_data(raw):
    # 1. Calculate 20 minutes directly in SECONDS (20 mins * 60 secs = 1200 seconds)
    t_max_seconds = 20 * 60 - 1
    
    # 2. Crop and OVERWRITE the local variable 'raw'
    raw = raw.copy().crop(tmax=t_max_seconds)

    # 3. Apply your case-insensitive 10-20 montage map
    raw.set_montage('standard_1020', match_case=False)

    annot_muscle, _ = annotate_muscle_zscore(
        raw, ch_type="eeg", threshold=5, min_length_good=0.2, filter_freq=[100, 120]  # Lowered to sit safely below your 125 Hz Nyquist ceiling!
    )
    
    raw.set_annotations(raw.annotations + annot_muscle)

    d1 = raw.copy().notch_filter(freqs=60).filter(l_freq=1, h_freq=40)

    return d1


v6 = preprocess_data(raw)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1e+02 - 1.2e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 100.00
- Lower transition bandwidth: 25.00 Hz (-6 dB cutoff frequency: 87.50 Hz)
- Upper passband edge: 120.00 Hz
- Upper transition bandwidth: 5.00 Hz (-6 dB cutoff frequency: 122.50 Hz)
- Filter length: 165 samples (0.660 s)

Setting up low-pass filter at 4 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 4.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 5.00 Hz)
- Filter length: 413 samples (1.652 s)

Filtering raw data 

In [30]:
ica6 = mne.preprocessing.ICA()
ica6.fit(v6.copy().pick('eeg'))
ica6.plot_components()

Fitting ICA to data using 24 channels (please be patient, this may take a while)
Omitting 6460 of 299751 (2.16%) samples, retaining 293291 (97.84%) samples.
Selecting by non-zero PCA components: 24 components
Fitting ICA took 3.5s.


[<MNEFigure size 1950x1780 with 20 Axes>, <MNEFigure size 780x260 with 4 Axes>]

In [ ]:
ica2.plot_sources(v2)

Creating RawArray with float64 data, n_channels=24, n_times=299751
    Range : 0 ... 299750 =      0.000 ...  1199.000 secs
Ready.


: 

In [31]:
cur = ica6.apply(v6.copy(),exclude=[4,10,12,14,15,20])

Applying ICA to Raw instance
    Transforming to ICA space (24 components)
    Zeroing out 6 ICA components
    Projecting back using 24 PCA components


In [ ]:
fv1 = cur    # exclude=[12,13]

In [14]:
fv2 = cur # exclude=[4,5,16]

In [18]:
fv3 = cur # exclude=[5,7,12]

In [23]:
fv4 = cur # exclude=[2,11,18,19]

In [27]:
fv5 = cur # exclude=[2,4,7,11,12,13,15,16]

In [32]:
fv6 = cur # exclude=[4,10,12,14,15,20]

In [33]:
import pandas as pd


rois = {
    'Frontal': [
        'FP1', 'FP2', 'F3', 'F4', 'F7', 'F8', 'AFZ', 'FZ'
    ],
    
    'Auditory': [
        'T7', 'T8', 'C3', 'C4', 'CZ', 'CPZ'
    ],

    'Gonzalez': [
        'T7', 'T8', 'P8', 'O2', 'F8', 'F7'
    ]
}



def extract_roi_power(clean_raw, participant_id, condition_name):
    segment_duration = 20.0  
    total_duration = clean_raw.times[-1]
    n_segments = int(total_duration // segment_duration)
    
    rows = []
    
    for seg in range(n_segments):
        tmin = seg * segment_duration
        tmax = tmin + segment_duration
        
        # Crop out this 20-second slice of data
        seg_data = clean_raw.copy().crop(tmin=tmin, tmax=tmax, verbose=False)
        
        # Compute PSD
        psd_obj = seg_data.compute_psd(method='welch', fmin=18, fmax=22, n_fft=1024, verbose=False)
        psds, freqs = psd_obj.get_data(return_freqs=True)
        idx_20hz = np.argmin(np.abs(freqs - 20))
        
        global_20hz_power = psds[:, idx_20hz].mean()
        
        for roi_name, channels in rois.items():
            ch_indices = [psd_obj.ch_names.index(ch) for ch in channels if ch in psd_obj.ch_names]
            
            if ch_indices:
                roi_abs_power = psds[ch_indices, idx_20hz].mean()
                roi_relative_power = roi_abs_power / global_20hz_power
                
                rows.append({
                    'Participant': participant_id,
                    'Condition': condition_name,
                    'Segment_ID': seg + 1,  # Tracks the chronological timeline chunks
                    'ROI': roi_name,
                    'abs_power': roi_abs_power,
                    'relative_power': roi_relative_power  
                })
            
    return pd.DataFrame(rows)

# Run the updated pipeline
V01 = extract_roi_power(fv1, participant_id='V01', condition_name='Binaural')
V02 = extract_roi_power(fv2, participant_id='V02', condition_name='Binaural')
V03 = extract_roi_power(fv3, participant_id='V03', condition_name='Binaural')
V04 = extract_roi_power(fv4, participant_id='V04', condition_name='Binaural')
V05 = extract_roi_power(fv5, participant_id='V05', condition_name='Binaural')
V06 = extract_roi_power(fv6, participant_id='V06', condition_name='Binaural')
data_long = pd.concat([V01,V02,V03,V04,V05,V06], ignore_index=True)


/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 1024 is greater than input length  = 410, using nperseg = 410
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 1024 is greater than input length  = 568, using nperseg = 568
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 1024 is greater than input length  = 57, using nperseg = 57
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 1024 is greater than input length  = 91, using nperseg = 91
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 1024 is 

In [34]:
data_long.to_csv("open_power2.csv", index=False)

# repicate plot

In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def extract_split_power(clean_raw, participant_id, target_band=(4, 8)):
    """
    Splits the data into a 2-min baseline and 20-min session, 
    then extracts absolute power for the target channels.
    """
    # 1. Define channels from the paper
    channels = ['Fp1', 'AFz', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                'Cz', 'C4', 'T8', 'CPz', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'POz', 'O2']
    
    # Keep only available channels
    available_chs = [ch for ch in channels if ch in clean_raw.ch_names]
    clean_raw = clean_raw.copy().pick_channels(available_chs)
    
    # 2. Split the data
    # Baseline: 0 to 120 seconds
    base_raw = clean_raw.copy().crop(tmin=0, tmax=120, verbose=False)
    # Session: 120 to 1320 seconds (20 minutes)
    sess_raw = clean_raw.copy().crop(tmin=120, tmax=1199, verbose=False)
    
    sfreq = int(clean_raw.info['sfreq'])
    rows = []
    
    # 3. Helper function to compute Welch's PSD (1s window, 50% overlap per the paper)
    def get_power(raw_obj, condition_name):
        psd_obj = raw_obj.compute_psd(method='welch', n_fft=sfreq, n_per_seg=sfreq, 
                              n_overlap=0, fmin=1, fmax=35, verbose=False)
        psds, freqs = psd_obj.get_data(return_freqs=True)
        
        idx_min = np.argmin(np.abs(freqs - target_band[0]))
        idx_max = np.argmin(np.abs(freqs - target_band[1]))
        
        for i, ch in enumerate(psd_obj.ch_names):
            abs_power = psds[i, idx_min:idx_max].mean()
            rows.append({
                'Participant': participant_id,
                'Condition': condition_name,
                'Channel': ch,
                'Absolute_Power': abs_power
            })

    # Run for both splits
    get_power(base_raw, 'Baseline')
    get_power(sess_raw, 'Binaural')
    
    return pd.DataFrame(rows)

# Create the full dataframe (Assuming you are targeting the Theta band: 4-8 Hz)
# Change to (13, 30) if your beats were Beta beats!
target_freqs = (4, 8) 

df_all = pd.concat([
    extract_split_power(fv1, 'V01', target_band=target_freqs),
    extract_split_power(fv2, 'V02', target_band=target_freqs),
    extract_split_power(fv3, 'V03', target_band=target_freqs),
    extract_split_power(fv4, 'V04', target_band=target_freqs),
    extract_split_power(fv5, 'V05', target_band=target_freqs),
    extract_split_power(fv6, 'V06', target_band=target_freqs)
], ignore_index=True)

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 57, using nperseg = 57
  return _func(*args, **kwargs)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 91, using nperseg = 91
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 173, using nperseg = 173
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 71, using nperseg = 71
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 225, using nperseg = 225
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is great

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


...

/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 132, using nperseg = 132
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 71, using nperseg = 71
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 99, using nperseg = 99
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 170, using nperseg = 170
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is great

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 118, using nperseg = 118
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 57, using nperseg = 57
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 71, using nperseg = 71
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 31, using nperseg = 31
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 67, using nperseg = 67
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 155, using nperseg = 155
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 78, using nperseg = 78
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater than input length  = 74, using nperseg = 74
  return _func(*args, **kwargs)
/Users/johanandersen/miniconda3/envs/mne/lib/python3.14/site-packages/mne/time_frequency/psd.py:291: UserWarning: nperseg = 250 is greater

In [38]:
def compute_cohens_d(df):
    d_values = {}
    channels = df['Channel'].unique()
    
    for ch in channels:
        # Isolate power values across the 6 participants
        base_power = df[(df['Channel'] == ch) & (df['Condition'] == 'Baseline')]['Absolute_Power'].values
        sess_power = df[(df['Channel'] == ch) & (df['Condition'] == 'Binaural')]['Absolute_Power'].values
        
        # Calculate means and variances
        mean_diff = np.mean(sess_power) - np.mean(base_power)
        var_base = np.var(base_power, ddof=1)
        var_sess = np.var(sess_power, ddof=1)
        
        # Pooled Standard Deviation
        sd_pooled = np.sqrt((var_base + var_sess) / 2)
        
        # Absolute Cohen's d 
        d_values[ch] = np.abs(mean_diff / sd_pooled)
        
    return d_values

# Get the final dictionary of d-values
d_results = compute_cohens_d(df_all)

In [41]:
# 1. Prepare data for plotting
channels = ['Fp1', 'AFz', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
            'Cz', 'C4', 'T8', 'CPz', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'POz', 'O2']

# Ensure we only plot channels that actually exist in your dataset
plot_channels = [ch for ch in channels if ch in d_results]
d_scores = [d_results[ch] for ch in plot_channels]

x = np.arange(len(plot_channels))
width = 0.4  # Slightly wider since it's only one bar per channel

# 2. Setup Figure
fig, ax = plt.subplots(figsize=(14, 8))

# 3. Add the horizontal background spans for effect size magnitude
ax.axhspan(0, 0.2, color='#ffffcc', alpha=0.6, zorder=0)      # Negligible (Yellow)
ax.axhspan(0.2, 0.5, color='#ccffcc', alpha=0.6, zorder=0)    # Small (Green)
ax.axhspan(0.5, 0.8, color='#ffcccc', alpha=0.6, zorder=0)    # Medium (Red)
ax.axhspan(0.8, 1.0, color='#e6f2ff', alpha=0.6, zorder=0)    # Large (Cyan)

# 4. Plot the bars
ax.bar(x, d_scores, width, label='Binaural session', 
       color='#ef665d', edgecolor='gray', zorder=3)

# 5. Customize Axes and Labels
# "Effektstørrelse" er den korrekte danske term for Cohen's d "Magnitude"
ax.set_ylabel('Effektstørrelse', fontsize=14)
ax.set_xlabel('Kanaler', fontsize=14)
ax.set_title('Cohen d-værdier (Baseline vs Binaural)', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(plot_channels, rotation=45, fontsize=12)

# Set Y-axis to dynamically fit your highest d-value (min 1.0 for the paper's look)
y_max = max(1.0, max(d_scores) + 0.1)
ax.set_ylim([0, y_max])
ax.grid(axis='x', linestyle='-', alpha=0.3, zorder=1) 

# 6. Create Custom Legends
magnitude_elements = [
    Patch(facecolor='#e6f2ff', edgecolor='gray', label='Stor'),
    Patch(facecolor='#ffcccc', edgecolor='gray', label='Mellem'),
    Patch(facecolor='#ccffcc', edgecolor='gray', label='Lille'),
    Patch(facecolor='#ffffcc', edgecolor='gray', label='Ubetydelig')
]
leg1 = ax.legend(handles=magnitude_elements, title='Forskelle', 
                 title_fontproperties={'weight':'bold'}, loc='upper left', bbox_to_anchor=(0.02, 0.98))
ax.add_artist(leg1) 

ax.legend(title='Session', title_fontproperties={'weight':'bold'}, 
          loc='upper right', bbox_to_anchor=(0.98, 0.98))

plt.tight_layout()
plt.show()